# silver_curated_ops_full

Light-conformance passthroughs for the operational / supply-chain / supplier tables that the retail / hr / clickstream / weather notebooks don't touch:

| silver_raw          | silver_curated |
|---------------------|----------------|
| inventory           | inventory      |
| payments            | payment        |
| promotions          | promotion      |
| returns             | return         |
| reviews             | review         |
| shipments           | shipment       |
| suppliers           | supplier       |
| warehouses          | warehouse      |

These are mostly straight passthroughs (string trim, a handful of derived columns). Curated tables stay singular to match the rest of the silver_curated layer.

Re-runnable: full overwrite.

In [ ]:
# Parameters baked by deploy.ps1.
silver_raw_workspace_id     = ""
silver_raw_lakehouse_id     = ""
silver_curated_workspace_id = ""
silver_curated_lakehouse_id = ""

In [ ]:
from pyspark.sql import functions as F

for n, v in [
    ('silver_raw_workspace_id',     silver_raw_workspace_id),
    ('silver_raw_lakehouse_id',     silver_raw_lakehouse_id),
    ('silver_curated_workspace_id', silver_curated_workspace_id),
    ('silver_curated_lakehouse_id', silver_curated_lakehouse_id),
]:
    if not v:
        raise ValueError(f'{n} parameter is required')

src_base = f'abfss://{silver_raw_workspace_id}@onelake.dfs.fabric.microsoft.com/{silver_raw_lakehouse_id}/Tables/dbo'
tgt_base = f'abfss://{silver_curated_workspace_id}@onelake.dfs.fabric.microsoft.com/{silver_curated_lakehouse_id}/Tables/dbo'

def read_src(name):
    return spark.read.format('delta').load(f'{src_base}/{name}')

def write_tgt(df, name):
    (df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(f'{tgt_base}/{name}'))
    print(f'  wrote dbo.{name:10s} ({df.count():>7,} rows, {len(df.columns)} cols)')

## Tiny reference dims
Suppliers / warehouses / promotions are small lookups. Just trim strings and pass through.

In [ ]:
supplier = (read_src('suppliers')
    .withColumn('supplier_name', F.trim('supplier_name'))
    .withColumn('contact_email', F.lower(F.trim('contact_email'))))
write_tgt(supplier, 'supplier')

warehouse = (read_src('warehouses')
    .withColumn('warehouse_name', F.trim('warehouse_name'))
    .withColumn('city',  F.trim('city'))
    .withColumn('state', F.trim('state')))
write_tgt(warehouse, 'warehouse')

promotion = (read_src('promotions')
    .withColumn('promo_code', F.upper(F.trim('promo_code')))
    .withColumn('promo_name', F.trim('promo_name'))
    .withColumn('is_active',
        (F.current_timestamp() >= F.col('starts_at')) &
        (F.current_timestamp() <= F.col('ends_at')))
    .withColumn('usage_remaining',
        F.greatest(F.col('usage_limit') - F.col('times_used'), F.lit(0))))
write_tgt(promotion, 'promotion')

## Operational facts

In [ ]:
# inventory: snapshot grain (product, location). Derived stockout flag.
inventory = (read_src('inventory')
    .withColumn('quantity_available',
        F.greatest(F.col('quantity_on_hand') - F.col('quantity_reserved'), F.lit(0)))
    .withColumn('is_stockout', F.col('quantity_on_hand') <= F.lit(0))
    .withColumn('needs_reorder', F.col('quantity_on_hand') <= F.col('reorder_point')))
write_tgt(inventory, 'inventory')

# payment: 1 row per payment. Status -> is_settled flag.
payment = (read_src('payments')
    .withColumn('payment_method', F.lower(F.trim('payment_method')))
    .withColumn('status',         F.lower(F.trim('status')))
    .withColumn('is_settled',     F.col('status') == F.lit('completed'))
    .withColumn('processed_date', F.to_date('processed_at')))
write_tgt(payment, 'payment')

# return: 1 row per return. Reserved word -> bracket downstream.
ret = (read_src('returns')
    .withColumn('return_status', F.lower(F.trim('return_status')))
    .withColumn('is_completed',  F.col('return_status') == F.lit('completed'))
    .withColumn('requested_date', F.to_date('requested_at'))
    .withColumn('completed_date', F.to_date('completed_at'))
    .withColumn('days_to_complete',
        F.datediff(F.col('completed_date'), F.col('requested_date'))))
write_tgt(ret, 'return')

# review: 1 row per review. Add rating buckets.
review = (read_src('reviews')
    .withColumn('review_title', F.trim('review_title'))
    .withColumn('created_date', F.to_date('created_at'))
    .withColumn('rating_bucket',
        F.when(F.col('rating') <= 2, F.lit('detractor'))
         .when(F.col('rating') == 3, F.lit('neutral'))
         .otherwise(F.lit('promoter'))))
write_tgt(review, 'review')

# shipment: 1 row per shipment. Lead time + on-time flag.
shipment = (read_src('shipments')
    .withColumn('carrier', F.upper(F.trim('carrier')))
    .withColumn('status',  F.lower(F.trim('status')))
    .withColumn('shipped_date',   F.to_date('shipped_at'))
    .withColumn('delivered_date', F.to_date('delivered_at'))
    .withColumn('transit_days',
        F.datediff(F.col('delivered_date'), F.col('shipped_date')))
    .withColumn('is_delivered', F.col('delivered_at').isNotNull())
    .withColumn('is_on_time',
        F.col('delivered_at').isNotNull() &
        (F.to_date('delivered_at') <= F.col('estimated_delivery'))))
write_tgt(shipment, 'shipment')

## Sanity check

In [ ]:
for n in ('supplier','warehouse','promotion','inventory','payment','return','review','shipment'):
    df = spark.read.format('delta').load(f'{tgt_base}/{n}')
    print(f'  {n:10s} {df.count():>7,} rows, cols={df.columns}')